# 🚀 LIA - Vetorizador de Catálogo CATMAT/CATSERV

Gera embeddings BGE-M3 com GPU para o catálogo do governo.

**Fluxo:**
1. Upload do `catalogo_gov.csv`
2. Gerar embeddings com GPU (~30 min)
3. Baixar `catalogo_vetorizado.parquet`

In [ ]:
# 1. Instalar dependências
!pip install -q sentence-transformers pandas pyarrow

In [ ]:
# 2. Upload do CSV
from google.colab import files
uploaded = files.upload()  # Selecione catalogo_gov.csv
CSV_FILE = list(uploaded.keys())[0]
print(f"Arquivo: {CSV_FILE}")

In [ ]:
# 3. Carregar CSV
import pandas as pd

df = pd.read_csv(CSV_FILE, dtype={"codigo": str})
print(f"Total: {len(df)} itens")
print(f"Materiais: {len(df[df.tipo == 'material'])}")
print(f"Serviços: {len(df[df.tipo == 'servico'])}")
df.head()

In [ ]:
# 4. Carregar BGE-M3 na GPU
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

model = SentenceTransformer("BAAI/bge-m3", device=device)
print("Modelo carregado!")

In [ ]:
# 5. Gerar embeddings (batch 512 na GPU, ~30 min para 340K itens)
import numpy as np
from datetime import datetime

textos = df["descricao"].tolist()
print(f"Gerando embeddings para {len(textos)} textos...")
print(f"Início: {datetime.now().strftime('%H:%M:%S')}")

BATCH_SIZE = 512

embeddings = model.encode(
    textos,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    device=device,
)

print(f"\nFim: {datetime.now().strftime('%H:%M:%S')}")
print(f"Shape: {embeddings.shape}")
print(f"Dimensões: {embeddings.shape[1]}")

In [ ]:
# 6. Adicionar embeddings ao DataFrame e salvar como Parquet
df["embedding"] = list(embeddings)

OUTPUT_FILE = "catalogo_vetorizado.parquet"
df.to_parquet(OUTPUT_FILE, index=False)

import os
size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
print(f"Salvo: {OUTPUT_FILE} ({size_mb:.1f} MB)")
print(f"Itens: {len(df)}")
print(f"Colunas: {list(df.columns)}")

In [ ]:
# 7. Download do Parquet
files.download(OUTPUT_FILE)
print("Download iniciado! Coloque o arquivo em catalogo/ e rode:")
print("  python catalogo/import_parquet.py --reset")